In [ ]:
SELECT table.name
FROM information_schema.tables
WHERE table_type = 'BASE TABLE'

To undestand our analysis we find out we have a total of 30 tables in our database, these are already organized in fact and dimension tables

In [ ]:
select top 5 *
from factinternetsales;


Our analysis will moslty be based on this two columsns and as we can see, We have a total of around 120k records in total.

In [ ]:
select min(orderdate) as first_order_date,max(orderdate) as last_order_date
from factresellersales
UNION ALL
select min(orderdate) as first_order_date,max(orderdate) as last_order_date
from factinternetsales
;

And this is the current date range for our fact tables. 

In [ ]:
drop view if exists Orders_by_category;
GO
CREATE VIEW Orders_by_category AS
    select englishproductcategoryname as Category,englishproductsubcategoryname AS SubCategory,sum(OrderQuantity) as number_of_orders,sum(ExtendedAmount) as Revenue, sum(ExtendedAmount - TotalProductCost) as profit
    from factinternetsales
    LEFT JOIN dimproduct
    ON factinternetsales.productkey = dimproduct.ProductKey
    left join dimproductsubcategory
    on dimproduct.ProductSubcategoryKey = dimproductsubcategory.ProductSubcategoryKey
    left join dimproductcategory
    on dimproductsubcategory.productcategorykey = dimproductcategory.productcategorykey
    --where dimproduct.ProductSubcategoryKey is not null 
    and dimproductcategory.englishproductcategoryname is not null
    group by englishproductsubcategoryname,englishproductcategoryname;
    --group by englishproductcategoryname;
go
select Category,SubCategory,number_of_orders
from Orders_by_category
order by number_of_orders DESC;

here we can see, where our sales are coming from based categories and subcategories

In [ ]:
select category, subcategory, revenue
from Orders_by_category
order by revenue DESC

In [ ]:
select category, subcategory, profit
from Orders_by_category
order by profit DESC

In [ ]:
WITH sales_per_year AS (
    SELECT 
        YEAR(orderdate) AS order_year, 
        SUM(orderquantity) AS total_orders
    FROM factinternetsales
    GROUP BY YEAR(orderdate)
)
SELECT 
    total_orders,
    order_year,
   ROUND(
    (
        total_orders - LAG(total_orders) OVER (ORDER BY order_year ASC)
    )
    / CAST(LAG(total_orders) OVER (ORDER BY order_year ASC) AS DECIMAL(10,2))
    * 100 ,0) AS percentage_change
FROM sales_per_year;

GO

WITH b2b_sales_per_year AS (
    SELECT 
        YEAR(orderdate) AS order_year, 
        SUM(orderquantity) AS total_orders
    FROM factresellersales
    GROUP BY YEAR(orderdate)
)
SELECT 
    total_orders,
    order_year,
   ROUND(
    (
        total_orders - LAG(total_orders) OVER (ORDER BY order_year ASC)
    )
    / CAST(LAG(total_orders) OVER (ORDER BY order_year ASC) AS DECIMAL(10,2))
    * 100 ,0) AS percentage_change
FROM b2b_sales_per_year;

In [ ]:
select column_name
from information_schema.columns
where table_name = 'dimproduct';

In [ ]:
with top_10_orders as 
        (select top 10 EnglishProductName as Top_10_by_order, rank() over(order by sum(OrderQuantity) DESC) AS ranking
        FROM factinternetsales as fs
        LEFT JOIN dimproduct as dp on fs.ProductKey = dp.ProductKey
        group by EnglishProductName),
    
        
top_10_revenue as (select top 10 EnglishProductName as Top_10_by_revenue, rank() over(order by sum(ExtendedAmount) DESC) AS ranking
        FROM factinternetsales as fs
        LEFT JOIN dimproduct as dp on fs.ProductKey = dp.ProductKey
        group by EnglishProductName),


top_10_profit as (select top 10 EnglishProductName as Top_10_by_profit, rank() over(order by sum(ExtendedAmount - TotalProductCost) DESC) AS ranking
        FROM factinternetsales as fs
        LEFT JOIN dimproduct as dp on fs.ProductKey = dp.ProductKey
        group by EnglishProductName)
        


select Top_10_by_order,Top_10_by_revenue,Top_10_by_profit
from top_10_orders 
LEFT JOIN top_10_revenue on top_10_orders.ranking = top_10_revenue.ranking
LEFT JOIN top_10_profit on top_10_orders.ranking = top_10_profit.ranking
;

Here we can find some interesting insights, the products that are sold the most are accesories, but the bikes are the ones that accounts for the most profit and revenue, and theres is also a really strong correlation between revenue and profit. and for the top 10 products is an identical correlation. This is an analysis for the B2C or Internet sales. 

In [ ]:
select avg(ExtendedAmount) as AOV_B2C
from FactInternetSales;
GO
SELECT AVG(extendedAmount) as AOV_B2B
FROM FactResellerSales;

The Average Order Per Customer is significantly higher on the b2b channel as compared to the b2c channel as expected.

In [ ]:
Select Tier, count(CustomerKey) as Customers
from vwDimCustomer
GROUP BY Tier;

 This is the Distribuition of retail customers based on the amount spent, i generated this tiering based on a dynamic logic, when customers who have spent more than the 85% of customers are considered to be VIP

In [ ]:
SELECT
    CustomerKey,
    SalesOrderNumber,
    OrderDate
FROM FactInternetSales
WHERE OrderDate >= '2010-01-01'
  AND OrderDate < '2011-01-01'
ORDER BY CustomerKey ASC, OrderDate ASC

In [ ]:
SELECT tier,count(tier) as customers,CAST(SUM(extendedAmount) as INT) as revenue,
    SUM(extendedAmount) * 100 / sum(sum(extendedamount)) over() as Percentag
from vwDimCustomer
LEFT JOIN factinternetsales on vwDimCustomer.customerkey = factinternetsales.customerkey
group by tier;

There are currently more than 13500 customers in the vip tier, which is our highest tier reserved for customers who have spent more than 90% of the total amount spent by customers, even though is less than other tiers, it still accounts for more Than 50% of our total revenue for internet or B2C sales 

In [ ]:
select accounttype,sum(Amount) as Total
from DimAccount as da
left join FactFinance as ff on da.AccountKey = ff.AccountKey
where accounttype is NOT NULL
group by  accounttype


In [ ]:
select top 10 *
from FactFinance